# Tratamiento de datos y pipeline de modelado

Este notebook documenta de forma explicativa los pasos que sigue el proyecto para preparar los datos, entrenar modelos, evaluarlos y explotar el modelo final. El contenido esta alineado con la implementacion real en `src\data_loader.py`, `src\model_trainer.py`, `src\evaluator.py`, `src\tuning.py`, `trainer.py`, `src\predictor.py` y `src\api.py`.


## 1. Definicion del problema y objetivos

**Problema de negocio**  
Queremos predecir si una reserva hotelera sera cancelada antes de la fecha de entrada.

**Variable objetivo**  
La variable objetivo es `is_canceled`, donde:
- `0`: la reserva no se cancela.
- `1`: la reserva se cancela.

**Objetivo tecnico**  
Construir un pipeline de clasificacion binaria capaz de estimar la probabilidad de cancelacion usando informacion operativa de la reserva.

**Objetivo del proyecto**  
1. Preparar correctamente el dataset.
2. Evitar fugas de informacion.
3. Comparar varios algoritmos bajo el mismo preprocesado.
4. Elegir el mejor modelo con una metrica robusta.
5. Dejar el modelo listo para inferencia local, API y Streamlit.


In [ ]:

from pathlib import Path
import sys


# Define la ruta princilal del proyecto ../../entregable
sys.path.insert(0, str(Path.cwd().resolve().parent.parent))

from src import config

import pandas as pd
from IPython.display import display

print('Ruta del dataset:', config.DATA_PATH)
print('Variable objetivo:', config.TARGET_COLUMN)
print('Columnas de leakage:', config.LEAKAGE_COLUMNS)
print('Metrica principal:', config.PRIMARY_METRIC)

Path.cwd()

: 

## 2. Seleccion de datos y limpieza

El dataset se carga desde `data\raw\dataset_practica_final.csv`. El proyecto usa dos ideas clave durante la preparacion:

### 2.1. Por que se eliminan algunas columnas

En `src\config.py` se definen como columnas de leakage:
- `reservation_status`
- `reservation_status_date`

Estas variables se eliminan porque contienen informacion demasiado cercana o posterior al resultado real de la reserva. Mantenerlas haria que el modelo aprenda una pista artificial sobre la cancelacion y sobreestime su rendimiento.

### 2.2. Por que se conservan el resto de variables

Se mantienen variables que estan disponibles en el contexto de negocio cuando la reserva ya existe o puede ser gestionada, por ejemplo:
- anticipacion de la reserva (`lead_time`)
- datos temporales de llegada
- duracion de la estancia
- numero de adultos, ninos y bebes
- segmento de mercado, canal de distribucion y tipo de cliente
- historial del cliente
- importe medio diario (`adr`)
- solicitudes especiales o necesidad de parking

### 2.3. Como se limpian los nulos

El proyecto **no elimina filas por tener nulos**. En su lugar, aplica imputacion dentro del pipeline:
- variables numericas -> `SimpleImputer(strategy='median')`
- variables categoricas -> `SimpleImputer(strategy='most_frequent')`

Este enfoque evita perder registros y hace que el mismo tratamiento se aplique tanto en entrenamiento como en inferencia.


In [3]:
from src.data_loader import load_raw_data, prepare_features

df = load_raw_data()
print('Dimensiones originales:', df.shape)
display(df.head())

null_summary = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)
null_summary = null_summary[null_summary > 0]
display(null_summary.to_frame(name='nulos'))

X, y = prepare_features(df)
print('Dimensiones tras eliminar target y leakage:')
print('X =', X.shape)
print('y =', y.shape)
print('Columnas eliminadas:', config.LEAKAGE_COLUMNS + [config.TARGET_COLUMN])


Dimensiones originales: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


,nulos
company,112593
agent,16340
country,488
children,4


Dimensiones tras eliminar target y leakage:
X = (119390, 29)
y = (119390,)
Columnas eliminadas: ['reservation_status', 'reservation_status_date', 'is_canceled']


In [4]:
from src.data_loader import build_preprocessor

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

print('Numero de variables numericas:', len(numeric_features))
print('Numero de variables categoricas:', len(categorical_features))
print('\nPrimeras variables numericas:', numeric_features[:10])
print('\nPrimeras variables categoricas:', categorical_features[:10])

preprocessor = build_preprocessor(X)
preprocessor


Numero de variables numericas: 19
Numero de variables categoricas: 10

Primeras variables numericas: ['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest']

Primeras variables categoricas: ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

### 2.4. Separacion train/test

La separacion se hace con `train_test_split` usando:
- `test_size = 0.2`
- `random_state = 42`
- `stratify = y`

La estratificacion es importante porque preserva la proporcion de cancelaciones entre entrenamiento y prueba, lo que hace mas fiable la evaluacion.


In [ ]:
from src.data_loader import split_data

X_train, X_test, y_train, y_test = split_data(X, y)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('Tasa de cancelacion global :', round(y.mean(), 4))
print('Tasa de cancelacion train  :', round(y_train.mean(), 4))
print('Tasa de cancelacion test   :', round(y_test.mean(), 4))


X_train: (95512, 29)
X_test : (23878, 29)
Tasa de cancelacion global : 0.3704
Tasa de cancelacion train  : 0.3704
Tasa de cancelacion test   : 0.3704


## 3. Seleccion del algoritmo

El proyecto compara varios clasificadores sobre el mismo preprocesado para que la comparacion sea justa:
- Regresion Logistica
- Arbol de Decision
- Random Forest
- Gradient Boosting o XGBoost si esta disponible
- Red neuronal multicapa (MLP)

### Por que tiene sentido esta seleccion

- **Regresion Logistica**: baseline interpretable y rapido.
- **Arbol de Decision**: captura reglas no lineales de forma sencilla.
- **Random Forest**: robusto con datos tabulares y mezcla de relaciones complejas.
- **Gradient Boosting / XGBoost**: suele rendir muy bien en tabular.
- **MLP**: aporta una referencia basada en red neuronal.

Todos los modelos se entrenan como un `Pipeline(preprocessor -> classifier)`. Asi se garantiza que el preprocesado es exactamente el mismo en entrenamiento, evaluacion e inferencia.


In [ ]:
from src.data_loader import build_data_bundle
from src.model_trainer import ModelTrainer

bundle = build_data_bundle()
trainer = ModelTrainer(bundle, quick_mode=True, skip_neural_net=False)

estimators = trainer._get_estimators()
list(estimators.keys())


['logistic_regression',
 'decision_tree',
 'random_forest',
 'gradient_boosting',
 'neural_network']

## 4. Entrenamiento y parametrizacion del algoritmo

En `src\model_trainer.py` cada modelo se ajusta dentro de un pipeline de `scikit-learn` con dos bloques:
1. `preprocessor`
2. `classifier`

### Parametrizacion base

Algunos hiperparametros relevantes definidos en el proyecto son:
- `LogisticRegression(max_iter=1000)`
- `DecisionTreeClassifier(max_depth=8)`
- `RandomForestClassifier(n_estimators=50 en quick / 200 en modo normal)`
- `XGBClassifier` o `GradientBoostingClassifier` como boosting
- `MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True)`

### Tuning

El ajuste de hiperparametros solo se activa si el mejor modelo base es `random_forest`. Para ello se usa `GridSearchCV` o `RandomizedSearchCV` sobre el pipeline completo. Las busquedas modifican hiperparametros como:
- `classifier__n_estimators`
- `classifier__max_depth`
- `classifier__min_samples_split`
- `classifier__min_samples_leaf`
- `classifier__max_features`


In [7]:
trained = trainer.train_all()
print('Modelos entrenados:', list(trained.models.keys()))
print('Artefactos generados:')
for name, path in trained.model_paths.items():
    print(f'- {name}: {path}')


Modelos entrenados: ['logistic_regression', 'decision_tree', 'random_forest', 'gradient_boosting', 'neural_network']
Artefactos generados:
- logistic_regression: C:\Users\Usuario\OneDrive\Pontia\Modulo 5 - DeepLearning\2.Proyecto Final de Módulo\entregable\models\logistic_regression.pkl
- decision_tree: C:\Users\Usuario\OneDrive\Pontia\Modulo 5 - DeepLearning\2.Proyecto Final de Módulo\entregable\models\decision_tree.pkl
- random_forest: C:\Users\Usuario\OneDrive\Pontia\Modulo 5 - DeepLearning\2.Proyecto Final de Módulo\entregable\models\random_forest.pkl
- gradient_boosting: C:\Users\Usuario\OneDrive\Pontia\Modulo 5 - DeepLearning\2.Proyecto Final de Módulo\entregable\models\gradient_boosting.pkl
- neural_network: C:\Users\Usuario\OneDrive\Pontia\Modulo 5 - DeepLearning\2.Proyecto Final de Módulo\entregable\models\neural_network.pkl


In [8]:
from src.tuning import tune_pipeline

# Ejemplo opcional de tuning del Random Forest ya entrenado.
# Puede tardar varios minutos segun el hardware.

# tuned_model, tuning_result = tune_pipeline(
#     pipeline=trained.models['random_forest'],
#     X_train=bundle.X_train,
#     y_train=bundle.y_train,
#     method='randomized',
#     cv=3,
#     n_iter=15,
#     scoring=config.PRIMARY_METRIC,
# )
# tuning_result


## 5. Evaluacion del modelo

La evaluacion se realiza en `src\evaluator.py`. El proyecto usa:
- **Metrica principal**: `roc_auc`
- **Metricas secundarias**: `accuracy`, `precision`, `recall` y `f1`

### Por que AUC-ROC es la metrica principal

La cancelacion de reservas puede presentar cierto desbalance entre clases. AUC-ROC es una metrica adecuada porque evalua la capacidad del modelo para separar clases a distintos umbrales, en lugar de depender solo de un punto de corte fijo.

### Artefactos de evaluacion

Ademas de la tabla de metricas, el pipeline genera:
- `outputs\metrics_summary.csv`
- `outputs\roc_curves.png`
- `outputs\confusion_<modelo>.png`
- `outputs\feature_importance_random_forest.png` si aplica


In [9]:
from src.evaluator import Evaluator

evaluator = Evaluator(bundle.y_test, bundle.X_test, trained.models)
metrics_df = evaluator.evaluate()
display(metrics_df)

best_row = metrics_df.iloc[0]
print('Mejor modelo segun', config.PRIMARY_METRIC, ':', best_row['model'])
print('Valor de la metrica:', round(float(best_row[config.PRIMARY_METRIC]), 6))


,model,accuracy,precision,recall,f1,roc_auc
2,random_forest,0.893291,0.895590,0.805879,0.848369,0.957454
4,neural_network,0.872770,0.853095,0.793103,0.822006,0.944677
1,decision_tree,0.829592,0.785168,0.743358,0.763691,0.909176
3,gradient_boosting,0.831560,0.862904,0.648276,0.740349,0.908756
0,logistic_regression,0.819206,0.812707,0.665235,0.731613,0.896183


Mejor modelo segun roc_auc : random_forest
Valor de la metrica: 0.957454


In [10]:
metrics_path = Path(config.METRICS_PATH)
if metrics_path.exists():
    display(pd.read_csv(metrics_path))
else:
    print('Todavia no existe outputs/metrics_summary.csv. Ejecuta la celda anterior o el trainer.')


,model,accuracy,precision,recall,f1,roc_auc
0,random_forest,0.893291,0.895590,0.805879,0.848369,0.957454
1,neural_network,0.872770,0.853095,0.793103,0.822006,0.944677
2,decision_tree,0.829592,0.785168,0.743358,0.763691,0.909176
3,gradient_boosting,0.831560,0.862904,0.648276,0.740349,0.908756
4,logistic_regression,0.819206,0.812707,0.665235,0.731613,0.896183


## 6. Explotacion del modelo

Una vez elegido el mejor modelo, el proyecto lo guarda como `models\best_model.pkl`. Este artefacto no guarda solo el estimador; tambien conserva el nombre del modelo ganador.

### Formas de explotacion disponibles en el proyecto

1. **Uso local en Python** con `src\predictor.py`.
2. **API REST** con FastAPI (`POST /predict`, `POST /train`, `GET /evaluate`).
3. **Interfaz Streamlit** en `app.py`.
4. **Tracking de ejecuciones** con MLflow y persistencia de runs en SQLite.

### Idea clave de explotacion

El payload de entrada se convierte en un `DataFrame` y el pipeline completo aplica automaticamente el mismo tratamiento de datos usado en entrenamiento. Esto evita inconsistencias entre desarrollo y produccion.


In [ ]:
from src.predictor import load_best_model, predict_from_payload

best_model_file = Path(config.MODELS_DIR) / 'best_model.pkl'
print('Existe best_model.pkl:', best_model_file.exists())

if best_model_file.exists():
    model_name, model = load_best_model()
    print('Modelo cargado:', model_name)

    sample_payload = bundle.X_test.iloc[0].to_dict()
    prediction = predict_from_payload(sample_payload)
    prediction
else:
    print('Primero debes entrenar y guardar un best_model.pkl.')


Existe best_model.pkl: True


c:\Users\Usuario\OneDrive\Pontia\Modulo 5 - DeepLearning\env-pontia-ml\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.4.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Usuario\OneDrive\Pontia\Modulo 5 - DeepLearning\env-pontia-ml\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Usuario\OneDrive\Pontia\Modulo 5 - DeepLearning\env-pontia-ml\Lib\site-packages\sklearn\base.py:463: Inconsi

Modelo cargado: random_forest


AttributeError: 'SimpleImputer' object has no attribute '_fill_dtype'

In [ ]:
api_endpoints = {
    'health': 'GET /health',
    'train': 'POST /train',
    'predict': 'POST /predict',
    'evaluate': 'GET /evaluate',
    'runs': 'GET /runs',
}
api_endpoints


## Conclusiones

El tratamiento de datos del proyecto se apoya en cuatro decisiones tecnicas importantes:

1. **Eliminar leakage** antes de entrenar.
2. **Imputar nulos dentro del pipeline** para no perder registros y mantener consistencia.
3. **Separar train/test con estratificacion** para evaluar correctamente.
4. **Guardar el pipeline completo** para reutilizar exactamente el mismo tratamiento en explotacion.

Con este enfoque, el proyecto no solo entrena modelos, sino que deja preparado un flujo reproducible desde la limpieza de datos hasta la puesta en produccion del mejor clasificador.
